## **Calendar: Spatial and Temporal GTFS data**

### **i. Libraries**

We begin by importing essential libraries for temporal and geospatial data analysis.
- **NumPy, pandas**: for numerical and tabular data manipulation.
- **Matplotlib, Seaborn**: for visualization.
- **GeoPandas**: for geospatial operations.
- **Missingno**: for quick visual inspection of missing values.
- **Counter**: to count occurrences of items in a collection

In [18]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import pandas as pd
import missingno as msno
import geopandas as gpd

### **ii. Data**

We load the previously prepared dataset that already contains both spatial and temporal attributes — including service accessibility and Gini-based equity indicators.

In [19]:
wards_full_gdf = gpd.read_parquet('/home/dataopske/Desktop/jav/data/processed/wards_full_gdf(with_spatial_temporal_gini).parquet')
wards_full_gdf.isna().sum()

gid                          0
pop2009                      0
county                       0
subcounty                    0
ward                         0
uid                          0
scuid                        0
cuid                         0
geometry                     0
population                   0
constituency                 0
poverty_rate                 0
intersect_area               0
ward_area                    0
coverage_ratio               0
pop_served                   0
pct_access                   0
pop_density                  0
coverage_area_km2            0
pop_not_served               0
subcounty_gini               0
hour                         0
trips_per_hour               0
trips_per_1k_pop_per_hour    0
service_rank_overall         0
service_rank_by_hour         0
dtype: int64

No missing data. 
>This confirms that the dataset is clean and ready for downstream processes, such as adding calendar information for sequential or longitudinal analysis.

### **iii. Objectives:**
1. Clean the data for downstream processing.

2. Add a calendar component to enable sequential (day-by-day) or temporal aggregation (weekday/weekend, week-by-week).

> Adding a calendar helps simulate real-world scheduling, allowing us to extend analysis from hours (previous notebook) to days and weeks, a key step for operational and equity studies over time.

### **iv. Create a Monday-Sunday**

We create a simple daily calendar for the month of October 2025.
This will later allow us to join service data by date, distinguish weekdays from weekends, and perform trend or frequency analysis.

In [7]:
import pandas as pd

# Create date range
calendar = pd.DataFrame({
    'date': pd.date_range(start='2025-10-01', end='2025-10-31', freq='D')
})

# Add day of week (0=Monday, 6=Sunday)
calendar['weekday'] = calendar['date'].dt.day_name()
calendar['is_weekend'] = calendar['weekday'].isin(['Saturday', 'Sunday'])
calendar.head()


,date,weekday,is_weekend
0,2025-10-01,Wednesday,False
1,2025-10-02,Thursday,False
2,2025-10-03,Friday,False
3,2025-10-04,Saturday,True
4,2025-10-05,Sunday,True


**Interpretation**

- **date** → provides a sequential daily structure for merging or aggregating service data.
- **weekday** → allows comparison of weekday vs. weekend transit patterns.
- **is_weekend** → enables quick filtering for operational contrasts (e.g., peak vs. off-peak service).

This foundational calendar table will later be joined with temporal GTFS metrics to produce date-aware service distributions — bridging spatial, temporal, and socio-economic equity layers.

### **v. Merge Calendar with Ward-Level Data**

We now combine the ward-level geospatial dataset with the calendar table to create a full spatio-temporal structure.
This enables analysis of transit service patterns for every ward, for every date, allowing for longitudinal and time-series exploration.

In [8]:
# Your original data
df = wards_full_gdf.copy()

# Add a constant key to both for merge
df['key'] = 1
calendar['key'] = 1

# Cross join
df_with_calendar = pd.merge(df, calendar, on='key').drop('key', axis=1)


### **vi. Adjusting for Weekend Service Levels**

After creating the full ward × date × hour table, we now account for reduced transit service on weekends. In Nairobi, weekend service is typically lower due to reduced demand and informal route adjustments.

#### **1. Sort Data**

In [9]:
df_with_calendar = df_with_calendar.sort_values(['ward', 'date', 'hour']).reset_index(drop=True)


- Sorting ensures that subsequent operations respect ward → date → hour order, which is important for time-series consistency.

#### **2. Define Weekend Multipliers**

We use weekday vs. weekend multipliers to adjust trips_per_hour:
| Day      | Multiplier | Notes                                                        |
| -------- | ---------- | ------------------------------------------------------------ |
| Saturday | 0.7        | Central estimate; literature shows 60–80% of weekday service |
| Sunday   | 0.5        | Central estimate; literature shows 40–60% of weekday service |


Many cities, including Nairobi, see weekend service fall in roughly 40–80% of weekday peaks. Multipliers are conservative mid-range estimates based on ITDP and local observations.

#### **3. Adjusting Service for Weekend Days**


After creating a **ward × hour × date** table through a cross-join, we account for **reduced transit service on weekends**. In Nairobi, weekend service is typically lower due to reduced demand and informal route adjustments.

1. **Central multipliers** are defined based on literature and local observations:

   * **Saturday:** 0.7 of weekday service
   * **Sunday:** 0.5 of weekday service

2. **Apply multipliers:**

   * A new column `trips_per_hour_adj` is created by scaling the weekday trips according to the day of the week.
   * Weekday values remain unchanged.

3. **Recompute normalized metrics:**

   * `service_per_1k_pop_adj` – trips per hour per 1,000 people
   * `trips_per_person_per_hour_adj` – trips per hour per person

In [10]:
# Assume df_with_calendar is your ward-hour-date table after the cross-join
# and has columns: 'date','weekday','trips_per_hour','ward','hour',...

df = df_with_calendar.copy()

# central estimate multipliers
sat_mult = 0.7
sun_mult = 0.5

# apply multipliers (create new column so original stays)
def apply_weekend_multiplier(row, sat_mult=0.7, sun_mult=0.5):
    wd = row['weekday']
    if wd == 'Saturday':
        return row['trips_per_hour'] * sat_mult
    elif wd == 'Sunday':
        return row['trips_per_hour'] * sun_mult
    else:
        return row['trips_per_hour']

df['trips_per_hour_adj'] = df.apply(apply_weekend_multiplier, axis=1, sat_mult=sat_mult, sun_mult=sun_mult)

# recompute normalized metrics
df['service_per_1k_pop_adj'] = df['trips_per_hour_adj'] / df['population'] * 1000
df['trips_per_person_per_hour_adj'] = df['trips_per_hour_adj'] / df['population']


> This adjustment allows the dataset to **reflect realistic temporal variation** in service across the week while preserving population-normalized metrics for equity analysis.

#### **4. Sensitivity Analysis: Weekend Scenarios**

To account for **uncertainty in weekend service reductions**, we test multiple **scenarios**:

* **Candidate multipliers** for weekend days:

  * Scenario 1: Saturday 0.6, Sunday 0.4
  * Scenario 2: Saturday 0.7, Sunday 0.5 (**central estimate**)
  * Scenario 3: Saturday 0.8, Sunday 0.6

**Process:**

1. For each candidate, create a temporary copy of the ward-hour-date dataset.
2. Scale the `trips_per_hour` for Saturday and Sunday according to the candidate multipliers; weekday values remain unchanged.
3. Compute the **average trips per hour by weekday**.
4. Store the results in a list for inspection.

In [11]:
candidates = [
    {'sat':0.6, 'sun':0.4},
    {'sat':0.7, 'sun':0.5},  # central
    {'sat':0.8, 'sun':0.6},
]
# inspect results list to see how weekend means change
results = []
for c in candidates:
    tmp = df.copy()
    tmp['trips_per_hour_adj'] = tmp.apply(lambda r: r['trips_per_hour'] * (c['sat'] if r['weekday']=='Saturday' else (c['sun'] if r['weekday']=='Sunday' else 1.0)), axis=1)
    summary = tmp.groupby('weekday')['trips_per_hour_adj'].mean().reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
    results.append({'mult':c, 'weekday_means': summary.to_dict()})


> This approach allows us to **explore how different assumptions about weekend service affect average service levels** across the week and check the robustness of downstream metrics like normalized trips and service equity.

In [12]:
results

[{'mult': {'sat': 0.6, 'sun': 0.4},
  'weekday_means': {'Monday': 649.9294117647058,
   'Tuesday': 649.9294117647058,
   'Wednesday': 649.9294117647058,
   'Thursday': 649.9294117647058,
   'Friday': 649.9294117647058,
   'Saturday': 389.9576470588235,
   'Sunday': 259.9717647058824}},
 {'mult': {'sat': 0.7, 'sun': 0.5},
  'weekday_means': {'Monday': 649.9294117647058,
   'Tuesday': 649.9294117647058,
   'Wednesday': 649.9294117647058,
   'Thursday': 649.9294117647058,
   'Friday': 649.9294117647058,
   'Saturday': 454.9505882352941,
   'Sunday': 324.9647058823529}},
 {'mult': {'sat': 0.8, 'sun': 0.6},
  'weekday_means': {'Monday': 649.9294117647058,
   'Tuesday': 649.9294117647058,
   'Wednesday': 649.9294117647058,
   'Thursday': 649.9294117647058,
   'Friday': 649.9294117647058,
   'Saturday': 519.9435294117648,
   'Sunday': 389.9576470588235}}]

**Weekend Adjustment Summary**

We applied three sets of **weekend multipliers** to the weekday trips per hour to account for lower weekend service. The results show:

* **Weekdays (Monday–Friday)** remain unchanged at ~650 trips/hour.
* **Saturday** service drops proportionally to the multipliers (0.6 → ~390, 0.7 → ~455, 0.8 → ~520 trips/hour).
* **Sunday** service is further reduced (0.4 → ~260, 0.5 → ~325, 0.6 → ~390 trips/hour).

> These scenarios provide a **conservative range of weekend service levels** and help prepare the dataset for temporal modeling, ensuring realistic variations in service over the week.


### **v. Summary and Next Steps**

At this stage, we have **prepared a comprehensive ward-level dataset** that combines spatial and temporal transit service information, normalized by population, and adjusted for **weekend variations**. Each record now captures:

* **Spatial identifiers**: ward, subcounty, constituency, county, and geometry.
* **Demographics**: population, poverty rate, and coverage metrics.
* **Service metrics**: trips per hour, normalized trips per 1k population, ranks (overall and hourly), and adjusted weekend values.
* **Temporal context**: hour of day, date, weekday, and weekend indicator.

This structure provides a **complete ward × hour × date grid**, making it ready to integrate **trajectory-level mobility data** (e.g., from WorldMove) for sequential analysis.

> The adjusted service metrics and full calendar coverage will allow **LSTM models or other temporal approaches** to learn patterns in service availability over time and predict equity outcomes under different scenarios. This setup also enables exploration of **peak vs. off-peak disparities**, weekday-weekend differences, and the interplay between service and population distribution.

In [13]:
# Save data as parquet
df.to_parquet('/home/dataopske/Desktop/jav/data/processed/spatio_temporal_calendar.parquet', index=False)

In [22]:
pd.set_option('display.max_columns', None)
print(df.head(7))


    gid  pop2009   county              subcounty          ward          uid  \
0  2078  43168.0  Nairobi  Kamukunji  Sub County  Airbase Ward  t4Suo8Enc7T   
1  2078  43168.0  Nairobi  Kamukunji  Sub County  Airbase Ward  t4Suo8Enc7T   
2  2078  43168.0  Nairobi  Kamukunji  Sub County  Airbase Ward  t4Suo8Enc7T   
3  2078  43168.0  Nairobi  Kamukunji  Sub County  Airbase Ward  t4Suo8Enc7T   
4  2078  43168.0  Nairobi  Kamukunji  Sub County  Airbase Ward  t4Suo8Enc7T   
5  2078  43168.0  Nairobi  Kamukunji  Sub County  Airbase Ward  t4Suo8Enc7T   
6  2078  43168.0  Nairobi  Kamukunji  Sub County  Airbase Ward  t4Suo8Enc7T   

         scuid         cuid  \
0  qoLIT7y5f5c  jkG3zaihdSs   
1  qoLIT7y5f5c  jkG3zaihdSs   
2  qoLIT7y5f5c  jkG3zaihdSs   
3  qoLIT7y5f5c  jkG3zaihdSs   
4  qoLIT7y5f5c  jkG3zaihdSs   
5  qoLIT7y5f5c  jkG3zaihdSs   
6  qoLIT7y5f5c  jkG3zaihdSs   

                                            geometry     population  \
0  POLYGON ((263765.584 9860350.471, 263805.22 